## Temporal Cross-Validation – Expanding Window

Objetivo:
Evaluar modelos de predicción probabilística en Liga MX usando un esquema
temporal realista (train en temporadas pasadas, validación en la siguiente).

Motivación:
Evitar leakage temporal y obtener métricas comparables a producción.

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data" / "processed"
df = pd.read_csv(DATA_DIR / "prematch_seasons18-24.csv")

df = df.sort_values(["season", "date"]).reset_index(drop=True)

df.head()


,season,match_id,date,home_advantage,home_win,home_team,away_team,home_form,away_form,form_diff,home_momentum,away_momentum,momentum_diff
0,2018 A,0,2018-07-20,1,0,Atlas Guadalajara,Gallos Blancos,0.066667,0.066667,0.0,0.0,0.0,0.0
1,2018 A,1,2018-07-20,1,0,CD Veracruz,Pumas UNAM,0.000000,0.200000,-0.2,0.0,0.0,0.0
2,2018 A,3,2018-07-21,1,0,CF Pachuca,CF Monterrey,0.000000,0.200000,-0.2,0.0,0.0,0.0
3,2018 A,5,2018-07-21,1,1,Club Tijuana,Deportivo Guadalajara,0.200000,0.000000,0.2,0.0,0.0,0.0
4,2018 A,2,2018-07-21,1,1,Cruz Azul,Puebla FC,0.200000,0.000000,0.2,0.0,0.0,0.0


In [3]:
features = [
    "home_form",
    "away_form",
    "home_momentum",
    "away_momentum"
]

target = "home_win"

In [4]:
seasons = sorted(df["season"].unique())
print(seasons)

['2018 A', '2019 A', '2019 C', '2020 A', '2021 A', '2021 C', '2022 A', '2022 C', '2023 A', '2023 C', '2024 A', '2024 C']


In [5]:
splits = []

for i in range(1, len(seasons)):
    train_seasons = seasons[:i]
    val_season = seasons[i]

    train_idx = df[df["season"].isin(train_seasons)].index
    val_idx = df[df["season"] == val_season].index

    splits.append((train_idx, val_idx))
print(splits)

[(Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
       ...
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152],
      dtype='int64', length=153), Index([153, 154, 155, 156, 157, 158, 159, 160, 161, 162,
       ...
       314, 315, 316, 317, 318, 319, 320, 321, 322, 323],
      dtype='int64', length=171)), (Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
       ...
       314, 315, 316, 317, 318, 319, 320, 321, 322, 323],
      dtype='int64', length=324), Index([324, 325, 326, 327, 328, 329, 330, 331, 332, 333,
       ...
       467, 468, 469, 470, 471, 472, 473, 474, 475, 476],
      dtype='int64', length=153)), (Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
       ...
       467, 468, 469, 470, 471, 472, 473, 474, 475, 476],
      dtype='int64', length=477), Index([477, 478, 479, 480, 481, 482, 483, 484, 485, 486,
       ...
       620, 621, 622, 623, 624, 625, 626, 627, 628, 629],
      dtype='int64', length=153)), (Index([  0,   1,   2,   3,   4, 

In [6]:
results = []

for train_idx, val_idx in splits:
    X_train = df.loc[train_idx, features]
    y_train = df.loc[train_idx, target]

    X_val = df.loc[val_idx, features]
    y_val = df.loc[val_idx, target]

    model = LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=1000
    )

    model.fit(X_train, y_train)

    probs = model.predict_proba(X_val)[:, 1]

    results.append({
        "train_seasons": df.loc[train_idx, "season"].unique().tolist(),
        "val_season": df.loc[val_idx, "season"].iloc[0],
        "log_loss": log_loss(y_val, probs),
        "brier": brier_score_loss(y_val, probs)
    })


c:\Users\USER\Python\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\USER\Python\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\USER\Python\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'pen

In [7]:
results_df = pd.DataFrame(results)
results_df


,train_seasons,val_season,log_loss,brier
0,[2018 A],2019 A,0.320952,0.096134
1,"[2018 A, 2019 A]",2019 C,0.252930,0.072307
2,"[2018 A, 2019 A, 2019 C]",2020 A,0.273640,0.081997
3,"[2018 A, 2019 A, 2019 C, 2020 A]",2021 A,0.246968,0.073198
4,"[2018 A, 2019 A, 2019 C, 2020 A, 2021 A]",2021 C,0.242546,0.073343
5,"[2018 A, 2019 A, 2019 C, 2020 A, 2021 A, 2021 C]",2022 A,0.265140,0.082842
6,"[2018 A, 2019 A, 2019 C, 2020 A, 2021 A, 2021 ...",2022 C,0.268001,0.086716
7,"[2018 A, 2019 A, 2019 C, 2020 A, 2021 A, 2021 ...",2023 A,0.240811,0.072271
8,"[2018 A, 2019 A, 2019 C, 2020 A, 2021 A, 2021 ...",2023 C,0.220117,0.065790
9,"[2018 A, 2019 A, 2019 C, 2020 A, 2021 A, 2021 ...",2024 A,0.240844,0.074878


In [8]:
print(df.loc[val_idx, "season"].nunique())

1


In [9]:
max(df.loc[train_idx, "season"]) < df.loc[val_idx, "season"].iloc[0]

True

In [10]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss
import numpy as np
import pandas as pd


In [11]:
C_values = [0.01, 0.1, 1.0, 10.0]

In [12]:
results = []

for split_id, (train_idx, val_idx) in enumerate(splits):

    X_train = df.loc[train_idx, features]
    y_train = df.loc[train_idx, target]

    X_val = df.loc[val_idx, features]
    y_val = df.loc[val_idx, target]

    # --- KFold INTERNO solo sobre train ---
    inner_kf = KFold(n_splits=5, shuffle=True, random_state=42)

    C_scores = []

    for C in C_values:
        fold_losses = []

        for inner_train_idx, inner_val_idx in inner_kf.split(X_train):
            X_tr = X_train.iloc[inner_train_idx]
            y_tr = y_train.iloc[inner_train_idx]

            X_va = X_train.iloc[inner_val_idx]
            y_va = y_train.iloc[inner_val_idx]

            model = LogisticRegression(
                penalty="l2",
                C=C,
                solver="lbfgs",
                max_iter=1000
            )

            model.fit(X_tr, y_tr)
            probs = model.predict_proba(X_va)[:, 1]

            fold_losses.append(log_loss(y_va, probs))

        C_scores.append({
            "C": C,
            "mean_log_loss": np.mean(fold_losses)
        })

    C_df = pd.DataFrame(C_scores)
    best_C = C_df.loc[C_df["mean_log_loss"].idxmin(), "C"]

    # --- Entrenamos modelo final con el mejor C ---
    final_model = LogisticRegression(
        penalty="l2",
        C=best_C,
        solver="lbfgs",
        max_iter=1000
    )

    final_model.fit(X_train, y_train)
    final_probs = final_model.predict_proba(X_val)[:, 1]

    results.append({
        "split": split_id,
        "train_seasons": df.loc[train_idx, "season"].unique().tolist(),
        "val_season": df.loc[val_idx, "season"].iloc[0],
        "best_C": best_C,
        "log_loss": log_loss(y_val, final_probs),
        "brier": brier_score_loss(y_val, final_probs)
    })

c:\Users\USER\Python\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\USER\Python\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\USER\Python\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'pen

In [13]:
results_df = pd.DataFrame(results)
results_df

,split,train_seasons,val_season,best_C,log_loss,brier
0,0,[2018 A],2019 A,10.0,0.259476,0.082766
1,1,"[2018 A, 2019 A]",2019 C,10.0,0.190820,0.059721
2,2,"[2018 A, 2019 A, 2019 C]",2020 A,10.0,0.258634,0.077426
3,3,"[2018 A, 2019 A, 2019 C, 2020 A]",2021 A,10.0,0.217367,0.068300
4,4,"[2018 A, 2019 A, 2019 C, 2020 A, 2021 A]",2021 C,10.0,0.218384,0.067892
5,5,"[2018 A, 2019 A, 2019 C, 2020 A, 2021 A, 2021 C]",2022 A,10.0,0.253505,0.080946
6,6,"[2018 A, 2019 A, 2019 C, 2020 A, 2021 A, 2021 ...",2022 C,10.0,0.262045,0.087737
7,7,"[2018 A, 2019 A, 2019 C, 2020 A, 2021 A, 2021 ...",2023 A,10.0,0.218084,0.067916
8,8,"[2018 A, 2019 A, 2019 C, 2020 A, 2021 A, 2021 ...",2023 C,10.0,0.199678,0.061413
9,9,"[2018 A, 2019 A, 2019 C, 2020 A, 2021 A, 2021 ...",2024 A,10.0,0.231118,0.074183


In [14]:
results_df["best_C"].value_counts()

best_C
10.0    11
Name: count, dtype: int64

In [15]:
pd.Series(
    final_model.coef_[0],
    index=features
).sort_values()


away_form       -5.910510
away_momentum   -3.003133
home_momentum    6.401648
home_form        9.021641
dtype: float64

En este bloque no se optimizan hiperparámetros ni se modifican features.
El objetivo es evaluar generalización temporal real.

In [16]:
seasons = (
    df["season"]
    .sort_values()
    .unique()
)
seasons


array(['2018 A', '2019 A', '2019 C', '2020 A', '2021 A', '2021 C',
       '2022 A', '2022 C', '2023 A', '2023 C', '2024 A', '2024 C'],
      dtype=object)

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss
import pandas as pd

FEATURES = [
    "home_form",
    "away_form",
    "home_momentum",
    "away_momentum"
]

TARGET = "home_win"

results = []


In [19]:
for i in range(1, len(seasons)):
    
    train_seasons = seasons[:i]
    test_season = seasons[i]
    
    train_df = df[df["season"].isin(train_seasons)]
    test_df  = df[df["season"] == test_season]
    
    # seguridad mínima
    if len(test_df) < 50:
        continue
    model = LogisticRegression(
        C=10,
        solver="lbfgs",
        max_iter=1000
    )
    
    model.fit(train_df[FEATURES], train_df[TARGET])
    y_true = test_df[TARGET]
    y_prob = model.predict_proba(test_df[FEATURES])[:, 1]
    
    ll = log_loss(y_true, y_prob)
    bs = brier_score_loss(y_true, y_prob)
    results.append({
        "eval_season": test_season,
        "train_seasons": ", ".join(train_seasons),
        "n_train": len(train_df),
        "n_test": len(test_df),
        "log_loss": ll,
        "brier": bs
    })




In [20]:
 
results_df = pd.DataFrame(results)
results_df

,eval_season,train_seasons,n_train,n_test,log_loss,brier
0,2019 A,2018 A,153,171,0.259476,0.082766
1,2019 C,"2018 A, 2019 A",324,153,0.190820,0.059721
2,2020 A,"2018 A, 2019 A, 2019 C",477,153,0.258634,0.077426
3,2021 A,"2018 A, 2019 A, 2019 C, 2020 A",630,153,0.217367,0.068300
4,2021 C,"2018 A, 2019 A, 2019 C, 2020 A, 2021 A",783,152,0.218384,0.067892
5,2022 A,"2018 A, 2019 A, 2019 C, 2020 A, 2021 A, 2021 C",935,153,0.253505,0.080946
6,2022 C,"2018 A, 2019 A, 2019 C, 2020 A, 2021 A, 2021 C...",1088,152,0.262045,0.087737
7,2023 A,"2018 A, 2019 A, 2019 C, 2020 A, 2021 A, 2021 C...",1240,150,0.218084,0.067916
8,2023 C,"2018 A, 2019 A, 2019 C, 2020 A, 2021 A, 2021 C...",1390,155,0.199678,0.061413
9,2024 A,"2018 A, 2019 A, 2019 C, 2020 A, 2021 A, 2021 C...",1545,153,0.231118,0.074183


El esquema de Expanding Window revela una curva de aprendizaje no monótona,
influenciada por cambios estructurales del fútbol (COVID, rotación de plantillas).
El modelo muestra recuperación consistente tras periodos de drift,
indicando robustez más que sobreajuste.